# Process and merge population files

In [178]:
# import argparse
import re
import sys
from pathlib import Path

import pandas as pd

In [179]:
def rename_period_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename data columns to their extracted period and sum columns that
    share the same period label.  Non-data columns are kept as-is.
    """
    id_cols = ["organisationunitid", "organisationunitname", "organisationunitcode"]
    rename_map: dict[str, str] = {}
    no_period: list[str] = []

    for col in df.columns:
        if col in id_cols:
            continue
        period = " ".join(col.split()[-2:])
        if period:
            rename_map[col] = period
        else:
            no_period.append(col)

    if no_period:
        print(
            f"[warning] Could not extract a period from {len(no_period)} column(s); "
            f"they will be dropped:\n  " + "\n  ".join(no_period)
        )

    # Keep only id cols + mappable cols
    keep_cols = id_cols + list(rename_map.keys())
    df = df[keep_cols].rename(columns=rename_map)
    
    remaining_items = [item for item in list(df.columns) if item not in set(id_cols)]
    new_list = id_cols + remaining_items
    result = df[new_list]

    return result

In [180]:
df = pd.read_csv('population.csv')
df.pop("organisationunitdescription")

0      à¤«à¤•à¥�à¤¤à¤¾à¤™à¥�à¤²à¥�à¤™à¥�à¤— à¤—à¤¾à¤‰...
1      à¤®à¤¿à¤•à¥�à¤µà¤¾à¤–à¥‹à¤²à¤¾ à¤—à¤¾à¤‰à¤�à¤ª...
2      à¤®à¥‡à¤°à¤¿à¤™à¥�à¤—à¤¦à¥‡à¤¨ à¤—à¤¾à¤‰à¤�à¤ª...
3      à¤®à¥ˆà¤µà¤¾à¤–à¥‹à¤²à¤¾ à¤—à¤¾à¤‰à¤�à¤ªà¤¾à¤²...
4      à¤†à¤ à¤°à¤¾à¤ˆ à¤¤à¥�à¤°à¤¿à¤µà¥‡à¤£à¥€ à¤—à¤...
                             ...                        
748    à¤®à¤¾à¤¹à¤¾à¤•à¤¾à¤²à¥€ à¤¨à¤—à¤°à¤ªà¤¾à¤²à¤¿...
749    à¤²à¤¾à¤²à¤�à¤¾à¤¡à¥€ à¤—à¤¾à¤‰à¤�à¤ªà¤¾à¤²à¤¿...
750    à¤ªà¥�à¤¨à¤°à¥�à¤µà¤¾à¤¸ à¤¨à¤—à¤°à¤ªà¤¾à¤²à¤¿...
751       à¤¬à¥‡à¤²à¥Œà¤°à¥€ à¤¨à¤—à¤°à¤ªà¤¾à¤²à¤¿à¤•à¤¾
752    à¤¬à¥‡à¤²à¤¡à¤¾à¤�à¤¡à¥€ à¤—à¤¾à¤‰à¤�à¤ªà¤¾à¤²...
Name: organisationunitdescription, Length: 753, dtype: str

In [181]:
# ---- Collapse period columns per level independently --------------------
print("Extracting and collapsing period columns...")
df_raw = rename_period_columns(df)

period_cols = [c for c in df_raw.columns if c not in ["organisationunitid", "organisationunitname", "organisationunitcode"]]
print(f"  {len(period_cols)} unique period(s) found: {', '.join(period_cols[:5])}" + "...")

Extracting and collapsing period columns...
  96 unique period(s) found: March 2024, February 2025, September 2022, March 2021, September 2019...


In [182]:
df_raw.columns = pd.to_datetime(df_raw.columns, format='%B %Y', errors = "coerce")

In [183]:
df_raw.head(3)

,NaT,NaT,NaT,2024-03-01,2025-02-01,2022-09-01,2021-03-01,2019-09-01,2022-02-01,2020-11-01,...,2024-11-01,2023-09-01,2019-07-01,2024-04-01,2021-11-01,2020-12-01,2023-02-01,2025-06-01,2023-08-01,2025-01-01
0,TnT6m8FZEet,10101 Phaktanlung Rural Municipality,10101,11197,11197,11951,11928,12283,11951,12305,...,11197,11197,12283,11197,11928,12305,11197,11197,11197,11197
1,gIhVZulvEsV,10102 Mikwakhola Rural Municipality,10102,7628,7628,8023,7995,9324,8023,9329,...,7628,7628,9324,7628,7995,9329,7628,7628,7628,7628
2,dcsJaeSXHa6,10103 Meringden Rural Municipality,10103,11198,11198,12086,12046,13743,12086,13789,...,11198,11198,13743,11198,12046,13789,11198,11198,11198,11198


In [184]:
time_cols = [c for c in df_raw.columns if c not in ["organisationunitid", "organisationunitname", "organisationunitcode"]]
df_time = df_raw[time_cols].T.reset_index().resample("YE", on="index").mean().T.reset_index(drop=True)

In [185]:
df_clean = pd.concat([df[["organisationunitid", "organisationunitname", "organisationunitcode"]], df_time], axis = 1)

In [186]:
df_clean

,organisationunitid,organisationunitname,organisationunitcode,2018-12-31 00:00:00,2019-12-31 00:00:00,2020-12-31 00:00:00,2021-12-31 00:00:00,2022-12-31 00:00:00,2023-12-31 00:00:00,2024-12-31 00:00:00,2025-12-31 00:00:00
0,TnT6m8FZEet,10101 Phaktanlung Rural Municipality,10101,12261.0,12283.0,12305.0,11928.0,11951.0,11197.0,11197.0,11197.0
1,gIhVZulvEsV,10102 Mikwakhola Rural Municipality,10102,9311.0,9324.0,9329.0,7995.0,8023.0,7628.0,7628.0,7628.0
2,dcsJaeSXHa6,10103 Meringden Rural Municipality,10103,13705.0,13743.0,13789.0,12046.0,12086.0,11198.0,11198.0,11198.0
3,zce1OaOO6G0,10104 Maiwakhola Rural Municipality,10104,11262.0,11272.0,11288.0,10370.0,10395.0,9730.0,9730.0,9730.0
4,zGGY6Sw9Uuw,10105 Aatharai Tribeni Rural Municipality,10105,14041.0,14065.0,14089.0,12289.0,12298.0,11665.0,11665.0,11665.0
...,...,...,...,...,...,...,...,...,...,...,...
748,Vaz2akuQgf5,70905 Dodharachadani Municipality,70905,44765.0,45539.0,46309.0,43596.0,44311.0,43645.0,43645.0,43645.0
749,NXKDqdR7CUd,70906 Laljhadi Rural Municipality,70906,25882.0,26330.0,26786.0,25337.0,25757.0,23686.0,23686.0,23686.0
750,uyEaMlGY93o,70907 Punarbas Municipality,70907,61003.0,62020.0,63055.0,61785.0,62787.0,63262.0,63262.0,63262.0
751,xTWW1v7S7qX,70908 Belouri Municipality,70908,61291.0,62345.0,63412.0,54048.0,54869.0,54456.0,54456.0,54456.0


In [187]:
df_clean[df_clean["organisationunitname"].astype(str).str.contains('Sani', na=False)]

,organisationunitid,organisationunitname,organisationunitcode,2018-12-31 00:00:00,2019-12-31 00:00:00,2020-12-31 00:00:00,2021-12-31 00:00:00,2022-12-31 00:00:00,2023-12-31 00:00:00,2024-12-31 00:00:00,2025-12-31 00:00:00
641,lfvPvTzR3Kt,60802 Sanibheri Rural Municipality,60802,23722.0,23939.0,24160.0,24822.0,25143.0,25576.0,25576.0,25576.0


In [188]:
df_clean['district_code'] = df_clean["organisationunitcode"].astype(str).str[:3]

In [189]:
df_dists = df_clean.groupby(["district_code"]).sum().reset_index().drop(columns=["organisationunitid", "organisationunitname", "organisationunitcode"])

In [190]:
df_dists.head(4)

,district_code,2018-12-31 00:00:00,2019-12-31 00:00:00,2020-12-31 00:00:00,2021-12-31 00:00:00,2022-12-31 00:00:00,2023-12-31 00:00:00,2024-12-31 00:00:00,2025-12-31 00:00:00
0,101,130395.0,130649.0,130890.0,120366.0,120455.0,114187.0,114187.0,114187.0
1,102,156583.0,156064.0,155531.0,159468.0,161292.0,154839.0,154839.0,154839.0
2,103,103060.0,102517.0,101969.0,104720.0,105343.0,102606.0,102606.0,102606.0
3,104,151153.0,151414.0,151657.0,140989.0,141505.0,134442.0,134442.0,134442.0


In [191]:
df_clean.to_csv("pop_muni.csv", index=False)
df_dists.to_csv("pop_dist.csv", index=False)

print(f"\nSaved:")
print(f"  {"pop_muni.csv"}")
print(f"  {"pop_dist.csv"}")
print("Done.")


Saved:
  pop_muni.csv
  pop_dist.csv
Done.
